In [ ]:
# Import necessary libraries
#%pip install git+https://github.com/IOHprofiler/iohclustering.git
#%pip install modcma

from iohclustering.utils import get_dataset
from iohclustering.cluster_baseline_problems import CLUSTER_BASELINE_DATASETS

from iohclustering import create_cluster_problem, general_cluster_metric, get_problem
import ioh
import numpy as np
import os


## Blade Function check how it works with clustering

In [ ]:
import pandas as pd
df_Km = pd.read_csv("kmeans_baseline.csv")

df_min = df_Km.groupby(['dataset', 'k']).agg('min')
df_min.to_csv("KMeansCompare.csv")

In [33]:
# IMPORTANT to use ollama pull qwen3-coder:30b

In [ ]:
# %pip install virtualenv
eg_prompts_list =  ["""An example of such code (a simple random search) for an IOH *clustering* objective is as follows.
Note: for clustering problems. Use the type of algorithm in the name of the program.

```python
import numpy as np

class RandomSearch:
    def __init__(self, budget, dim, seed=None):
        self.budget = budget
        self.dim = dim
        self.rng = np.random.default_rng(seed)

    def __call__(self, func):
        f_opt = np.inf
        x_opt = None

        lb = np.asarray(func.bounds.lb, dtype=float)
        ub = np.asarray(func.bounds.ub, dtype=float)

        while func.state.evaluations < self.budget:
            x = self.rng.uniform(lb, ub, size=self.dim)
            f = func(x)

            if f < f_opt:
                f_opt = f
                x_opt = x.copy()

        return f_opt, x_opt```""", 
"""An example of such code NeighborhoodAdaptiveDE for an IOH *clustering* objective is as follows.
Note: for clustering problems. Use the type of algorithm in the name of the program.

```python
import numpy as np
class NeighborhoodAdaptiveDE:
    def __init__(self, budget=2000, dim=None, seed=None):
        self.budget = budget
        self.dim = dim
        self.seed = seed

        self.success_f = []
        self.success_cr = []
        self.best_fitness_history = []

        self.pop_size = 50
        self.adapt_freq = 50
        self.stagnation_threshold = 1000
        self.learning_rate = 0.1
        self.neighborhood_size = 5
        self.f = 0.5
        self.cr = 0.9

    def __call__(self, func):
        np.random.seed(self.seed)
        f_opt = np.Inf
        x_opt = None
        self.population = np.random.uniform(func.bounds.lb, func.bounds.ub, size=(self.pop_size, self.dim))
        self.fitness = np.array([func(x) for x in self.population])
        self.evals = self.pop_size
        self.last_improvement = 0

        best_idx = np.argmin(self.fitness)
        f_opt = self.fitness[best_idx]
        x_opt = self.population[best_idx]
        self.best_fitness_history.append(f_opt)

        while self.evals < self.budget:
            for i in range(self.pop_size):
                neighbors = np.random.choice(np.arange(self.pop_size), self.neighborhood_size, replace=False)
                best_neighbor_idx = neighbors[np.argmin(self.fitness[neighbors])]

                idxs = np.random.choice(np.arange(self.pop_size), 2, replace=False)
                x1, x2 = self.population[idxs]

                v = self.population[best_neighbor_idx] + self.f * (x1 - x2)
                v = np.clip(v, func.bounds.lb, func.bounds.ub)

                u = np.copy(self.population[i])
                j_rand = np.random.randint(self.dim)
                for j in range(self.dim):
                    if np.random.rand() < self.cr or j == j_rand:
                        u[j] = v[j]

                f_new = func(u)
                self.evals += 1

                if f_new < self.fitness[i]:
                    self.success_f.append(self.f)
                    self.success_cr.append(self.cr)
                    self.fitness[i] = f_new
                    self.population[i] = u

                    if f_new < f_opt:
                        f_opt = f_new
                        x_opt = u
                        self.best_fitness_history.append(f_opt)
                        self.last_improvement = self.evals

            if self.evals % self.adapt_freq == 0:
                if self.success_f:
                    self.f = (1 - self.learning_rate) * self.f + self.learning_rate * np.mean(self.success_f)
                    self.cr = (1 - self.learning_rate) * self.cr + self.learning_rate * np.mean(self.success_cr)
                    self.f = np.clip(self.f, 0.1, 0.9)
                    self.cr = np.clip(self.cr, 0.1, 1.0)
                    self.success_f = []
                    self.success_cr = []

            if self.evals - self.last_improvement > self.stagnation_threshold:
                self.population = np.random.uniform(func.bounds.lb, func.bounds.ub, size=(self.pop_size, self.dim))
                self.fitness = np.array([func(x) for x in self.population])
                self.evals += self.pop_size
                best_idx = np.argmin(self.fitness)
                f_opt = self.fitness[best_idx]
                x_opt = self.population[best_idx]
                self.last_improvement = self.evals
                self.best_fitness_history.append(f_opt)

        return f_opt, x_opt```""", 
"""An example of such code a particle swarm optimiser for an IOH *clustering* objective is as follows.
Note: for clustering problems. Use the type of algorithm in the name of the program.

```python
import numpy as np

class ParticleSwarmOptimization:
    def __init__(self, budget=10000, dim=None, seed=None):
        self.budget = budget
        self.dim = dim
        self.rng = np.random.default_rng(seed)

        self.s = 50
        self.inertia = 0.5
        self.psi_p = 1.5
        self.psi_g = 1.5

    def __call__(self, func):
        positions = self.rng.uniform(func.bounds.lb, func.bounds.ub, size=(self.s, self.dim))
        domain = np.abs(np.array(func.bounds.ub) - np.array(func.bounds.lb))
        velocity = self.rng.uniform(-domain, domain, size=(self.s, self.dim))

        f = np.array([func(x) for x in positions])
        best_known_positions = positions.copy()
        best_known_values = f.copy()
        idx = np.argmin(f)
        f_opt, x_opt = f[idx], positions[idx, :]

        while func.state.evaluations + self.s <= self.budget:
            rp = self.rng.uniform(size=(self.s, 1))
            rg = self.rng.uniform(size=(self.s, 1))
            velocity = (
                (self.inertia * velocity)
                + (self.psi_p * rp * (best_known_positions - positions))
                + (self.psi_g * rg * (x_opt - positions))
            )

            positions += velocity
            f = np.array([func(x) for x in positions])
            improvements = f < best_known_values
            best_known_positions[improvements] = positions[improvements]
            best_known_values[improvements] = f[improvements]

            idy = np.argmin(f)
            if f[idy] < f_opt:
                f_opt = f[idy]
                x_opt = positions[idy]

            if abs(func.optimum.y - func.state.current_best.y) < 1e-8:
                break

        return f_opt, x_opt```""", 

"""An example of such code a UMDA solution for an IOH *clustering* objective is as follows.
Note: for clustering problems.

```python
import numpy as np
import math

class UMDA:
    def __init__(self, budget, dim, seed=None):
        self.budget = budget
        self.dim = dim
        self.rng = np.random.default_rng(seed)
        self.population_size = 50

    def __call__(self, func):
        n = self.dim
        s = self.population_size

        pop = self.rng.integers(0, 2, size=(s, n), dtype=np.uint8)
        fitness = np.empty(s, dtype=float)
        for i in range(s):
            fitness[i] = func(pop[i])

        best_idx = int(np.argmax(fitness))
        best_x = pop[best_idx].copy()
        best_f = float(fitness[best_idx])

        mu = s // 2
        idx_sorted = np.argsort(-fitness)
        parents = pop[idx_sorted[:mu]]

        while func.state.evaluations + s <= self.budget:
            counts = parents.sum(axis=0)
            p = np.clip(2.0 * counts / s, 1.0 / n, 1.0 - 1.0 / n)

            pop = (self.rng.random((s, n)) < p).astype(np.uint8)
            fitness = np.empty(s, dtype=float)
            for i in range(s):
                if func.state.evaluations >= self.budget:
                    return best_x, best_f
                fitness[i] = func(pop[i])
                if fitness[i] > best_f:
                    best_f, best_x = fitness[i], pop[i].copy()

            if best_f >= func.optimum.y:
                break

            idx_sorted = np.argsort(-fitness)
            parents = pop[idx_sorted[:mu]]

        return best_f, best_x```"""]

In [ ]:
from iohblade.experiment import Experiment
from iohblade.llm import Ollama_LLM
from iohblade.methods import LLaMEA
import os
from iohblade.loggers import ExperimentLogger
# from iohblade import Problem, Solution, wrap_problem
import numpy as np
from pathlib import Path
import os
import logging

from IOHClusteringAdapter import IOHClusteringProblemAdapter  # <-- key change

# (optional) reduce joblib multi-proc weirdness in notebooks
# os.environ["JOBLIB_MULTIPROCESSING"] = "0"


llm = Ollama_LLM("qwen3-coder:30b")

method = LLaMEA(
    llm,
    budget=200,          # LLaMEA algorithm-count budget
    name="LLaMEA",
    # example_prompts = eg_prompts_list, # warm start
    n_parents=4,
    n_offspring=8,    # How many children should I have? 
    elitism=True, # Important 
    minimization=False, 
)

methods = [method]

FIDS = range(1, 3)
K_ALL = [2, 3, 5, 10]

train_fid = range(1, 9)
test_fid  = range(9, 11)

# keeping the instances simple for now as it will be my first time running over the entire dataset
IID = 1

train_instances = [(fid, IID, k) for fid in train_fid for k in K_ALL]
test_instances  = [(fid, IID, k) for fid in test_fid for k in K_ALL]


# While you are assigning the training and test instances here you need to reset these to be the actual data points in your python file 

problem = IOHClusteringProblemAdapter(
    training_instances=train_instances,
    test_instances=test_instances,
    name="CLUSTERING",
    base_budget=25,
    budget_scale_with_dim=True,
    eval_timeout=120,
    full_ioh_log=False,
    ioh_dir="",  # or "ioh_logs"
)



logger = ExperimentLogger("/local/4077644-Anna/results/mutation_5_runs_elitism_on")
problems = [problem] # our dummy problem

experiment = Experiment(
    methods=methods,
    problems=problems,
    runs=5,
    show_stdout=True,     # LLaMEA redirects stdout internally; your debug_eval.log is the truth
    exp_logger=logger,
    budget=200,           # must be >= LLaMEA_method.budget, ideally a bit higher
)

experiment()


KeyboardInterrupt: 

In [ ]:
import os
import requests

requests.post(
    "https://api.pushover.net/1/messages.json",
    data={
        "token": "",
        "user": "",
        "title": "Mutation Prompts Done",
        "message": "5 runs of mutation prompts with eltism."
    },
    timeout=10,
)